## Naive Bayes Classifier

In [1]:
fill_questionmarks = 0
laplace_smoothing = 1
use_log = True

In [ ]:
%pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
congressional_voting_records = fetch_ucirepo(id=105)

# data (as pandas dataframes)
X = congressional_voting_records.data.features
y = congressional_voting_records.data.targets

# metadata
print(congressional_voting_records.metadata)

# variable information
print(congressional_voting_records.variables)

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
# pd.concat([X, y], axis=1).to_csv("data.csv")

Fill NaN (?)

In [ ]:
if fill_questionmarks == 0:
    X.fillna("unsure", inplace=True)
else:
    for col in X.columns:
        X[col].fillna(X[col].mode()[0], inplace=True)

Shuffle

In [7]:
df_shuffled = pd.concat([X, y], axis=1).sample(frac=1, random_state=1234).reset_index(drop=True)
X = df_shuffled.drop("Class", axis=1)
y = pd.DataFrame(df_shuffled["Class"])

80/20 split

In [ ]:
data = pd.concat([X, y], axis=1)

# Stratified sampling
train = data.groupby("Class", group_keys=False).apply(lambda x: x.sample(frac=0.8, random_state=1234))
X_train = train.drop("Class", axis=1)
y_train = pd.DataFrame(train["Class"])

test = data.loc[~data.index.isin(train.index)]
X_test = test.drop("Class", axis=1)
y_test = pd.DataFrame(test["Class"])

In [9]:
y_train.value_counts()

Class     
democrat      214
republican    134
Name: count, dtype: int64

In [10]:
y_test.value_counts()

Class     
democrat      53
republican    34
Name: count, dtype: int64

In [11]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Class     
democrat      0.614943
republican    0.385057
Name: proportion, dtype: float64
Class     
democrat      0.609195
republican    0.390805
Name: proportion, dtype: float64


Cross validation

In [12]:
def get_cross_validation_data(X, y, group_number=1, cross_groups=10):
    group_len = len(X) // cross_groups

    indices = np.arange(len(X))

    start = (group_number - 1) * group_len
    end = (group_number) * group_len
    mask = (indices >= start) & (indices < end)

    return X[~mask], y[~mask], X[mask], y[mask]

Bayes

In [13]:
df_current = pd.concat([X_train, y_train], axis=1)
df_classes_prob = y_train.value_counts() / len(y_train)
df_classes_prob

Class     
democrat      0.614943
republican    0.385057
Name: count, dtype: float64

In [14]:
df_republican = df_current[df_current["Class"] == "republican"].drop("Class", axis=1)
len(df_republican)

134

In [15]:
df_democrat = df_current[df_current["Class"] == "democrat"].drop("Class", axis=1)
len(df_democrat)

214

In [16]:
df_republican_prob = df_republican.apply(lambda col: col.value_counts() / len(df_republican))
df_republican_prob

,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-corporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa
n,0.783582,0.432836,0.828358,0.014925,0.037313,0.082090,0.716418,0.776119,0.865672,0.432836,0.820896,0.097015,0.126866,0.007463,0.843284,0.291045
unsure,0.022388,0.134328,0.029851,0.022388,0.022388,0.014925,0.044776,0.067164,0.022388,0.022388,0.059701,0.074627,0.067164,0.044776,0.082090,0.134328
y,0.194030,0.432836,0.141791,0.962687,0.940299,0.902985,0.238806,0.156716,0.111940,0.544776,0.119403,0.828358,0.805970,0.947761,0.074627,0.574627


In [17]:
laplace_smoothing = laplace_smoothing  # 0 should result in the same df as above!

In [18]:
df_republican_prob = df_republican.apply(
    lambda col: (col.value_counts() + laplace_smoothing) / (len(df_republican) + laplace_smoothing * col.nunique())
)
df_republican_prob

,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-corporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa
n,0.773723,0.430657,0.817518,0.021898,0.043796,0.087591,0.708029,0.766423,0.854015,0.430657,0.810219,0.102190,0.131387,0.014599,0.832117,0.291971
unsure,0.029197,0.138686,0.036496,0.029197,0.029197,0.021898,0.051095,0.072993,0.029197,0.029197,0.065693,0.080292,0.072993,0.051095,0.087591,0.138686
y,0.197080,0.430657,0.145985,0.948905,0.927007,0.890511,0.240876,0.160584,0.116788,0.540146,0.124088,0.817518,0.795620,0.934307,0.080292,0.569343


In [19]:
df_democrat_prob = df_democrat.apply(
    lambda col: (col.value_counts() + laplace_smoothing) / (len(df_democrat) + laplace_smoothing * col.nunique())
)
df_democrat_prob

,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-corporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa
n,0.373272,0.460829,0.115207,0.903226,0.741935,0.502304,0.207373,0.165899,0.221198,0.539171,0.488479,0.778802,0.682028,0.612903,0.313364,0.046083
unsure,0.041475,0.110599,0.036866,0.041475,0.050691,0.032258,0.041475,0.023041,0.069124,0.023041,0.046083,0.073733,0.050691,0.032258,0.059908,0.317972
y,0.585253,0.428571,0.847926,0.055300,0.207373,0.465438,0.751152,0.811060,0.709677,0.437788,0.465438,0.147465,0.267281,0.354839,0.626728,0.635945


In [20]:
assert (df_republican_prob >= 0).all().all()
assert (df_democrat_prob >= 0).all().all()

In [21]:
def predict_input(input_data, df_republican_prob, df_democrat_prob, df_classes_prob):
    republican_prob = df_classes_prob["republican"]
    democrat_prob = df_classes_prob["democrat"]

    for col, value in input_data.items():
        republican_prob *= df_republican_prob[col][value]
        democrat_prob *= df_democrat_prob[col][value]

    # print(republican_prob, democrat_prob)
    return "republican" if republican_prob > democrat_prob else "democrat"


def predict_input_log(input_data, df_republican_prob, df_democrat_prob, df_classes_prob):
    republican_prob = np.log(df_classes_prob["republican"])
    democrat_prob = np.log(df_classes_prob["democrat"])

    for col, value in input_data.items():
        republican_prob += np.log(df_republican_prob[col][value])
        democrat_prob += np.log(df_democrat_prob[col][value])

    # print(republican_prob, democrat_prob)
    return "republican" if republican_prob > democrat_prob else "democrat"

In [22]:
predict_input(X_train.iloc[0], df_republican_prob, df_democrat_prob, df_classes_prob)

'democrat'

In [23]:
predict_input_log(X_train.iloc[0], df_republican_prob, df_democrat_prob, df_classes_prob)

'democrat'

In [24]:
def Bayes(X_train, y_train, X_test, y_test, laplace_smoothing, use_log=False):

    df_current = pd.concat([X_train, y_train], axis=1)
    df_classes_prob = y_train.value_counts() / len(y_train)

    df_republican = df_current[df_current["Class"] == "republican"].drop("Class", axis=1)
    df_democrat = df_current[df_current["Class"] == "democrat"].drop("Class", axis=1)

    df_republican_prob = df_republican.apply(
        lambda col: (col.value_counts() + laplace_smoothing) / (len(df_republican) + laplace_smoothing * col.nunique())
    )

    df_democrat_prob = df_democrat.apply(
        lambda col: (col.value_counts() + laplace_smoothing) / (len(df_democrat) + laplace_smoothing * col.nunique())
    )

    function_to_use = predict_input_log if use_log else predict_input
    predictions = X_test.apply(
        lambda row: function_to_use(row, df_republican_prob, df_democrat_prob, df_classes_prob), axis=1
    )

    accuracy = np.mean(predictions == y_test["Class"]) * 100
    return accuracy, predictions

In [25]:
Bayes(X_train, y_train, X_test, y_test, laplace_smoothing=laplace_smoothing, use_log=use_log)

(91.95402298850574,
 4        democrat
 5        democrat
 6        democrat
 10     republican
 21       democrat
           ...    
 396      democrat
 402    republican
 403    republican
 408    republican
 420    republican
 Length: 87, dtype: object)

Cross fold

In [26]:
def one_fold_experiment(X, y, laplace_smoothing, use_log, group_number, cross_groups):
    X_train, y_train, X_test, y_test = get_cross_validation_data(
        X, y, group_number=group_number, cross_groups=cross_groups
    )
    accuracy, predictions = Bayes(X_train, y_train, X_test, y_test, laplace_smoothing, use_log)
    print(f"Accuracy Fold {group_number}: {accuracy:2f}%")

    return accuracy

In [27]:
def cross_validation_experiments(X, y, laplace_smoothing, use_log, cross_groups):
    accuracies = []
    for group_number in range(1, cross_groups + 1):
        accuracy = one_fold_experiment(X, y, laplace_smoothing, use_log, group_number, cross_groups)
        accuracies.append(accuracy)

    average_accuracy = np.mean(accuracies)
    standard_deviation = np.std(accuracies)
    return average_accuracy, standard_deviation

In [28]:
def experiment(laplace_smoothing=1, use_log=True, cross_groups=10):
    train_accuracy, train_predictions = Bayes(X_train, y_train, X_train, y_train, laplace_smoothing, use_log)
    print(f"1. Train set accuracy: \nAccuracy: {train_accuracy:2f}%\n")

    print(f"2. {cross_groups}-Fold Cross-Validation Results:")
    average_accuracy, standard_deviation = cross_validation_experiments(X, y, laplace_smoothing, use_log, cross_groups)
    print(f"\nAverage Accuracy: {average_accuracy:2f}%\nStandard Deviation: {standard_deviation:2f}\n")

    test_accuracy, test_predictions = Bayes(X_train, y_train, X_test, y_test, laplace_smoothing, use_log)
    print(f"3. Test set accuracy: \nAccuracy: {test_accuracy:2f}%\n")

    # return train_accuracy, average_accuracy, standard_deviation, test_accuracy

In [29]:
experiment(laplace_smoothing=1, use_log=False)

1. Train set accuracy: 
Accuracy: 89.942529%

2. 10-Fold Cross-Validation Results:
Accuracy Fold 1: 86.046512%
Accuracy Fold 2: 88.372093%
Accuracy Fold 3: 83.720930%
Accuracy Fold 4: 90.697674%
Accuracy Fold 5: 88.372093%
Accuracy Fold 6: 97.674419%
Accuracy Fold 7: 90.697674%
Accuracy Fold 8: 90.697674%
Accuracy Fold 9: 93.023256%
Accuracy Fold 10: 88.372093%

Average Accuracy: 89.767442%
Standard Deviation: 3.632674

3. Test set accuracy: 
Accuracy: 91.954023%



In [30]:
experiment(laplace_smoothing=0, use_log=False)

1. Train set accuracy: 
Accuracy: 90.229885%

2. 10-Fold Cross-Validation Results:
Accuracy Fold 1: 86.046512%
Accuracy Fold 2: 88.372093%
Accuracy Fold 3: 83.720930%
Accuracy Fold 4: 93.023256%
Accuracy Fold 5: 88.372093%
Accuracy Fold 6: 97.674419%
Accuracy Fold 7: 90.697674%
Accuracy Fold 8: 90.697674%
Accuracy Fold 9: 93.023256%
Accuracy Fold 10: 88.372093%

Average Accuracy: 90.000000%
Standard Deviation: 3.757092

3. Test set accuracy: 
Accuracy: 91.954023%



In [31]:
experiment(laplace_smoothing=1, use_log=True)

1. Train set accuracy: 
Accuracy: 89.942529%

2. 10-Fold Cross-Validation Results:
Accuracy Fold 1: 86.046512%
Accuracy Fold 2: 88.372093%
Accuracy Fold 3: 83.720930%
Accuracy Fold 4: 90.697674%
Accuracy Fold 5: 88.372093%
Accuracy Fold 6: 97.674419%
Accuracy Fold 7: 90.697674%
Accuracy Fold 8: 90.697674%
Accuracy Fold 9: 93.023256%
Accuracy Fold 10: 88.372093%

Average Accuracy: 89.767442%
Standard Deviation: 3.632674

3. Test set accuracy: 
Accuracy: 91.954023%

